Merge mobility data with European rankings, sending and receiving side.

In [ ]:
import pandas as pd
import json

In [ ]:
mobility = pd.read_csv("df_mobility_hicp.csv", index_col=0)
rankings = pd.read_csv("european_rankings.csv")
name_map = json.load(open("university_flat_mapping.json"))

Country aliases not covered by the ranking dataset.

In [ ]:
COUNTRY_ALIASES = {"czechia": "czech republic", "slovakia": "slovak republic", "türkiye": "turkey", "turkiye": "turkey"}

def norm_country(s):
    s = s.str.strip().str.casefold()
    return s.map(lambda c: COUNTRY_ALIASES.get(c, c))

def norm_institution(s):
    return s.map(name_map).fillna(s).str.strip().str.casefold()

In [ ]:
rank_key = rankings.assign(
    inst_key=norm_institution(rankings["Institution"]).astype("category"),
    country_key=norm_country(rankings["Country"]).astype("category"),
)[["inst_key", "country_key", "Year", "European Rank"]]

In [ ]:
mobility["send_inst_key"] = norm_institution(mobility["Sending Organization"]).astype("category")
mobility["send_country_key"] = norm_country(mobility["Sending Country"]).astype("category")
mobility["recv_inst_key"] = norm_institution(mobility["Receiving Organization"]).astype("category")
mobility["recv_country_key"] = norm_country(mobility["Receiving Country"]).astype("category")

Join on institution, country and year.

In [ ]:
merged = mobility.merge(
    rank_key.rename(columns={
        "inst_key": "send_inst_key", "country_key": "send_country_key",
        "Year": "Academic Year", "European Rank": "Sending Institution Rank",
    }),
    on=["send_inst_key", "send_country_key", "Academic Year"], how="left",
)

merged = merged.merge(
    rank_key.rename(columns={
        "inst_key": "recv_inst_key", "country_key": "recv_country_key",
        "Year": "Academic Year", "European Rank": "Receiving Institution Rank",
    }),
    on=["recv_inst_key", "recv_country_key", "Academic Year"], how="left",
)

merged = merged.drop(columns=["send_inst_key", "send_country_key", "recv_inst_key", "recv_country_key"])
merged.head(10)
merged.to_csv("mobility_rankings_merged.csv", index=False)